In [41]:
import networkx as nx
import pandas as pd

In [49]:
from convokit import Corpus, download
corpus = Corpus(filename=download("conversations-gone-awry-corpus"))
corpus.print_summary_stats()

Dataset already exists at /Users/caseygilbert/.convokit/saved-corpora/conversations-gone-awry-corpus
Number of Speakers: 8069
Number of Utterances: 30021
Number of Conversations: 4188


In [86]:

# Get convo df
convo_db = corpus.get_attribute_table("conversation", ["conversation_has_personal_attack", "page_title", "page_id", "pair_id", "conversation_has_personal_attack", "verified", "pair_verified", "annotation_year"])
convo_db.reset_index(inplace=True)

# Get utter df
utter_df = corpus.get_utterances_dataframe()

# Merge them to get speakers
full_utter = pd.merge(utter_df, convo_db, left_on='conversation_id', right_on='id', how='outer')

# And add speakers column to convo_db
convo_db['speakers'] = [[] for _ in range(len(convo_db))]

# Now loop through and give speakers to conversation db
for index, row in full_utter.iterrows():
    conversation_id = row["conversation_id"]
    if not row["speaker"] in convo_db.loc[convo_db['id'] == conversation_id]["speakers"].iloc[0]:
        convo_db.loc[convo_db['id'] == conversation_id]["speakers"].iloc[0].append(row["speaker"])

# Now get speakers df
speakers = corpus.get_speakers_dataframe()

In [87]:
# Build our bipartite network

sample_graph = nx.Graph()

for index in speakers.iterrows():
    sample_graph.add_node(index[0], bipartite=0, node_type="speaker")

for row, index in convo_db.iterrows():
    sample_graph.add_node(index["id"], bipartite=1, node_type="conversation", conversation_has_personal_attack=index["conversation_has_personal_attack"], page_title=index["page_title"], page_id=index["page_id"], pair_id=index["pair_id"], verified=index["verified"], pair_verified=index["pair_verified"], annotation_year=index["annotation_year"])
    for speaker in index["speakers"]:
        sample_graph.add_edge(speaker, index["id"], is_personal_attack=index["conversation_has_personal_attack"])


In [88]:
speakers_nodes = {n for n, d in sample_graph.nodes(data=True) if d["bipartite"] == 0}
conversations_nodes = {n for n, d in sample_graph.nodes(data=True) if d["bipartite"] == 1}

projected_graph = nx.bipartite.projected_graph(sample_graph, speakers_nodes)

print(f"Node count: {sample_graph.number_of_nodes()}")
print(f"Edge count: {sample_graph.number_of_edges()}")
print("\n  ")
print(f"Node count: {projected_graph.number_of_nodes()}")
print(f"Edge count: {projected_graph.number_of_edges()}")

Node count: 12257
Edge count: 13681

  
Node count: 8069
Edge count: 17407


In [92]:
type(convo_db["conversation_has_personal_attack"][0])

numpy.bool

In [ ]:
opposite_nodes = [n for n, d in sample_graph.nodes(data=True) if d.get("conversation_has_personal_attack") == True]
projected_G = nx.Graph()

for obj in conversations_nodes:
    # Get the specific characteristic of this opposite node
    characteristic = sample_graph.nodes[obj].get("conversation_has_personal_attack", "unknown")

    # Find all target nodes (Users) connected to this opposite node
    neighbors = list(sample_graph.neighbors(obj))

    # Connect all pairs of neighbors sharing this node
    for i in range(len(neighbors)):
        for j in range(i + 1, len(neighbors)):
            u, v = neighbors[i], neighbors[j]

            if not projected_G.has_node(u):
                projected_G.add_node(u)

            # If the edge already exists, update it; otherwise, create it
            if projected_G.has_edge(u, v):
                # Example: Append characteristic to a list of shared traits
                projected_G[u][v]["shared_via"].append(characteristic)
            else:
                # Initialize the edge with the characteristic attribute
                projected_G.add_edge(u, v, shared_via=[characteristic])


In [84]:
nx.write_gexf(sample_graph, "constructive_convo.gexf", version="1.2draft")
nx.write_gexf(projected_graph, "constructive_convo_projected.gexf", version="1.2draft")